In [0]:
# scores (project-defined heuristics, not medical metrics):

# 1. sleep_score = 70% duration score + 30% efficiency score
# 2. activity_score = step-count bands (demo thresholds)
# 3. heart_rate_score = resting-HR bands
# 4. recovery_score = 0.5 * sleep + 0.3 * activity + 0.2 * heart-rate (renormalised when a component is missing)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "health_and_habit_guardian"
GOLD_DAILY = f"{CATALOG}.gold.daily_health"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

# configurable heuristic parameters (override via Databricks widgets when run as a job).
TARGET_SLEEP_HOURS = 8.0
STEP_GOAL = 8000  # informational demo goal; scoring bands below are fixed for transparency
try:
    dbutils.widgets.text("target_sleep_hours", "8.0")
    dbutils.widgets.text("step_goal", "8000")
    TARGET_SLEEP_HOURS = float(dbutils.widgets.get("target_sleep_hours"))
    STEP_GOAL = int(dbutils.widgets.get("step_goal"))
except Exception:
    pass

print(f"TARGET_SLEEP_HOURS={TARGET_SLEEP_HOURS} STEP_GOAL={STEP_GOAL}")

TARGET_SLEEP_HOURS=8.0 STEP_GOAL=8000


In [0]:
# 1. Daily activity features.
# Sum makes the pipeline robust if a source later emits >1 record per user/day (synthetic seed data is already one row/day, so sum == the day value).
activity_daily = (
    spark.table(f"{CATALOG}.silver.activity")
    .filter(F.col("_quality_status") == "VALID")
    .groupBy("user_id", "date")
    .agg(
        F.sum("steps").cast("long").alias("steps"),
        F.sum("active_minutes").cast("long").alias("active_minutes"),
        F.sum("calories_burned").cast("long").alias("calories_burned"),
        F.avg("resting_hr").alias("resting_hr"),
    )
)

In [0]:
# 2. Daily heart-rate features (timestamped grain -> calendar date).
hr_daily = (
    spark.table(f"{CATALOG}.silver.heart_rate")
    .filter(F.col("_quality_status") == "VALID")
    .withColumn("date", F.to_date("timestamp"))
    .groupBy("user_id", "date")
    .agg(
        F.avg("heart_rate_bpm").alias("avg_heart_rate"),
        F.min("heart_rate_bpm").alias("min_heart_rate"),
        F.max("heart_rate_bpm").alias("max_heart_rate"),
    )
)

In [0]:
# 3. Daily sleep features (Silver sleep is already one row/night).
sleep_daily = (
    spark.table(f"{CATALOG}.silver.sleep")
    .filter(F.col("_quality_status") == "VALID")
    .select(
        "user_id",
        "date",
        F.col("sleep_duration_hours").alias("sleep_hours"),
        "deep_sleep_hours",
        "light_sleep_hours",
        "rem_sleep_hours",
        "awake_hours",
        "awakenings",
        "sleep_efficiency",
    )
)

In [0]:
# FULL OUTER join so a day with only some sources present is kept.
# 5. Missing-data flags so the future agent can say "no data" vs "bad night".
daily_health = (
    activity_daily
    .join(sleep_daily, ["user_id", "date"], "full")
    .join(hr_daily, ["user_id", "date"], "full")
)

daily_health = (
    daily_health
    .withColumn("activity_missing", F.col("steps").isNull())
    .withColumn("sleep_missing", F.col("sleep_hours").isNull())
    .withColumn("heart_rate_missing", F.col("avg_heart_rate").isNull())
)

In [0]:
# Sleep debt + rolling windows + trend features
daily_health = daily_health.withColumn(
    "sleep_debt_hours",
    F.when(
        F.col("sleep_hours").isNull(), F.lit(None).cast("double")
    ).otherwise(F.greatest(F.lit(0.0), F.lit(TARGET_SLEEP_HOURS) - F.col("sleep_hours"))),
)

w7 = Window.partitionBy("user_id").orderBy("date").rowsBetween(-6, 0)
w28 = Window.partitionBy("user_id").orderBy("date").rowsBetween(-27, 0)

daily_health = (
    daily_health
    .withColumn("rolling_7d_sleep_debt", F.sum("sleep_debt_hours").over(w7))
    .withColumn("rolling_7d_steps", F.avg("steps").over(w7))
    .withColumn("rolling_28d_steps", F.avg("steps").over(w28))
    .withColumn("steps_7d_avg", F.avg("steps").over(w7))
    .withColumn("sleep_7d_avg", F.avg("sleep_hours").over(w7))
    .withColumn("resting_hr_7d_avg", F.avg("resting_hr").over(w7))
    .withColumn("recovery_7d_avg", F.avg("recovery_score").over(w7))
)

# Baseline deviation: recent 7d vs previous 28d baseline.
daily_health = daily_health.withColumn(
    "activity_drop_pct",
    F.when(
        F.col("rolling_28d_steps").isNull() | (F.col("rolling_28d_steps") == 0),
        F.lit(None).cast("double"),
    ).otherwise(
        (F.col("rolling_28d_steps") - F.col("rolling_7d_steps")) / F.col("rolling_28d_steps")
    ),
)

daily_health = daily_health.withColumn("_processed_at", F.current_timestamp())

In [0]:
# Heuristic scores (0-100, project-defined, NOT medical advice).
# Sleep duration bands: 7-9h high, 6-7h moderate, 5-6h low, otherwise 30.
# Boundaries are exclusive on the lower side to avoid the overlapping between() trap.
duration_score = (
    F.when(F.col("sleep_hours").isNull(), F.lit(None).cast("double"))
    .when((F.col("sleep_hours") >= 7) & (F.col("sleep_hours") <= 9), F.lit(100.0))
    .when((F.col("sleep_hours") >= 6) & (F.col("sleep_hours") < 7), F.lit(75.0))
    .when((F.col("sleep_hours") >= 5) & (F.col("sleep_hours") < 6), F.lit(50.0))
    .otherwise(F.lit(30.0))
)

efficiency_score = (
    F.when(F.col("sleep_efficiency").isNull(), F.lit(None).cast("double"))
    .when(F.col("sleep_efficiency") >= 0.85, F.lit(100.0))
    .when(F.col("sleep_efficiency") >= 0.80, F.lit(85.0))
    .when(F.col("sleep_efficiency") >= 0.75, F.lit(70.0))
    .otherwise(F.lit(50.0))
)

daily_health = daily_health.withColumn("_duration_score", duration_score)
daily_health = daily_health.withColumn("_efficiency_score", efficiency_score)

daily_health = daily_health.withColumn(
    "sleep_score",
    F.when(
        F.col("_duration_score").isNull(), F.lit(None).cast("double")
    ).when(
        F.col("_efficiency_score").isNull(), F.col("_duration_score")
    ).otherwise(0.7 * F.col("_duration_score") + 0.3 * F.col("_efficiency_score")),
)

# Activity bands (demo thresholds, not a health requirement):
# 10k+ high, 7-10k good, 3-7k moderate, <3k low.
daily_health = daily_health.withColumn(
    "activity_score",
    F.when(F.col("steps").isNull(), F.lit(None).cast("double"))
    .when(F.col("steps") >= 10000, F.lit(100.0))
    .when(F.col("steps") >= 7000, F.lit(80.0))
    .when(F.col("steps") >= 3000, F.lit(60.0))
    .otherwise(F.lit(30.0)),
)

# Resting-HR bands (project heuristic): 55-68 high, fringe moderate, extremes low.
daily_health = daily_health.withColumn(
    "heart_rate_score",
    F.when(F.col("resting_hr").isNull(), F.lit(None).cast("double"))
    .when((F.col("resting_hr") >= 55) & (F.col("resting_hr") <= 68), F.lit(100.0))
    .when(
        ((F.col("resting_hr") >= 50) & (F.col("resting_hr") < 55))
        | ((F.col("resting_hr") > 68) & (F.col("resting_hr") <= 75)),
        F.lit(75.0),
    )
    .otherwise(F.lit(50.0)),
)

# Recovery = 0.5*sleep + 0.3*activity + 0.2*hr, renormalised over non-null parts
# so a day missing one source still gets a comparable score.
_num = (
    F.coalesce(0.5 * F.col("sleep_score"), F.lit(0.0))
    + F.coalesce(0.3 * F.col("activity_score"), F.lit(0.0))
    + F.coalesce(0.2 * F.col("heart_rate_score"), F.lit(0.0))
)
_den = (
    F.when(F.col("sleep_score").isNull(), F.lit(0.0)).otherwise(F.lit(0.5))
    + F.when(F.col("activity_score").isNull(), F.lit(0.0)).otherwise(F.lit(0.3))
    + F.when(F.col("heart_rate_score").isNull(), F.lit(0.0)).otherwise(F.lit(0.2))
)
daily_health = daily_health.withColumn(
    "recovery_score",
    F.when(_den == 0, F.lit(None).cast("double")).otherwise(_num / _den),
)

In [0]:
# Idempotent upsert on the business key (user_id, date).
final_cols = [
    "user_id", "date",
    "steps", "active_minutes", "calories_burned", "resting_hr",
    "sleep_hours", "deep_sleep_hours", "rem_sleep_hours", "light_sleep_hours",
    "awake_hours", "awakenings", "sleep_efficiency",
    "avg_heart_rate", "min_heart_rate", "max_heart_rate",
    "activity_missing", "sleep_missing", "heart_rate_missing",
    "activity_score", "sleep_score", "heart_rate_score", "recovery_score",
    "sleep_debt_hours", "rolling_7d_sleep_debt",
    "rolling_7d_steps", "rolling_28d_steps", "activity_drop_pct",
    "steps_7d_avg", "sleep_7d_avg", "resting_hr_7d_avg", "recovery_7d_avg",
    "_processed_at",
]

daily_health_final = daily_health.select(*final_cols)
daily_health_final.createOrReplaceTempView("daily_health_updates")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_DAILY}
USING DELTA
AS SELECT * FROM daily_health_updates WHERE 1 = 0
""")

cols = ", ".join(final_cols)
spark.sql(f"""
MERGE INTO {GOLD_DAILY} AS target
USING (
    SELECT {cols}
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY user_id, date
            ORDER BY _processed_at DESC NULLS LAST
        ) AS _rn
        FROM daily_health_updates
    )
    WHERE _rn = 1
) AS source
ON target.user_id = source.user_id AND target.date = source.date
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"SELECT COUNT(*) AS rows, COUNT(DISTINCT user_id) AS users, MIN(date), MAX(date) FROM {GOLD_DAILY}"))

rows,users,MIN(date),MAX(date)
160,4,2026-08-01,2026-09-09
